# MedImageInsight Baseline

image-only baseline notebook. Uses train-only CUI vocabulary with minimum frequency 5 and maximum vocabulary cap 2048 for evaluation. Retrieval ranking uses visual similarity only; CUI labels are used only after ranking for metrics.

In [ ]:
import os, re, json, math, random, time, warnings, shutil, subprocess, sys, base64, importlib.util
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore")


DATA_ROOT = os.environ.get("DATA_ROOT", "../data/roco_v2")

CFG = {
    "seed": 42,
    "random_seeds": [42, 123, 2025],
    "model_tag": "medimageinsight",
    "hf_repo_id": "lion-ai/MedImageInsights",
    "image_size": 512,


    "max_train_samples": None,
    "max_valid_samples": None,
    "max_test_samples": None,


    "epochs": 4,
    "head_lr": 2e-3,
    "weight_decay": 1e-4,
    "head_dropout": 0.20,
    "batch_size": 128,
    "embed_batch_size": 8,
    "projection_dim": 512,

    "num_workers": 0,
    "max_train_cuis": 2048,
    "min_cui_freq": 5,
    "eval_split": "test",
    "retrieval_pool": "test",
    "ks_main": [5, 10, 50],
    "ks_curve": list(range(1, 51)),
    "topk_max": 50,
    "eval_chunk_size": 256,
    "train": False,
    "reuse_cached_features": True,
    "use_trained_projection_for_retrieval": False,
}

OUT_DIR = os.environ.get("OUTPUT_DIR", os.path.join(os.environ.get("OUTPUT_ROOT", "../results"), "medimageinsight_roco_retrieval"))
os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
FEATURE_DIR = os.path.join(OUT_DIR, "cached_medimageinsight_features")
os.makedirs(FEATURE_DIR, exist_ok=True)
MODEL_CACHE_DIR = os.environ.get("MODEL_CACHE_DIR", "../models/medimageinsight_model_cache")
os.makedirs(MODEL_CACHE_DIR, exist_ok=True)

print("Model:", CFG["hf_repo_id"])
print("Output:", OUT_DIR)


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

try:
    from IPython.display import display
except Exception:
    display = lambda x: print(x.to_string() if hasattr(x, "to_string") else x)

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

def pip_install(cmd_list):
    print("Installing:", " ".join(cmd_list))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + cmd_list)


def find_data_root(preferred):
    if Path(preferred).exists():
        return preferred
    print("DATA_ROOT not found. Searching DATA_SEARCH_DIR ...")
    candidates = []
    for root, dirs, files in os.walk(os.environ.get("DATA_SEARCH_DIR", "../data")):
        if {"train_concepts.csv", "valid_concepts.csv", "test_concepts.csv"}.issubset(set(files)):
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError("Could not find folder with train/valid/test_concepts.csv")
    return candidates[0]

DATA_ROOT = find_data_root(DATA_ROOT)
print("Using DATA_ROOT:", DATA_ROOT)

CUI_RE = re.compile(r"C\d{7}", re.IGNORECASE)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def read_csv_flexible(path):
    for sep in [",", "\t", ";", "|"]:
        try:
            df = pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False)
            if df.shape[1] >= 1:
                return df
        except Exception:
            pass
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def build_image_map(split):
    split_dir = Path(DATA_ROOT) / split
    mp = {}
    for p in split_dir.rglob("*"):
        if p.suffix.lower() in IMG_EXTS:
            mp[p.stem] = str(p)
            mp[p.name] = str(p)
    return mp

def infer_id_col(df):
    for word in ["image", "file", "filename", "name", "id", "uid"]:
        for c in df.columns:
            if word in c.lower():
                return c
    return df.columns[0]

def extract_cuis_from_row(row):
    txt = " ".join([str(x) for x in row.values])
    return sorted(set([x.upper() for x in CUI_RE.findall(txt)]))

def load_split(split):
    csv_path = Path(DATA_ROOT) / f"{split}_concepts.csv"
    manual_path = Path(DATA_ROOT) / f"{split}_concepts_manual.csv"
    path = csv_path if csv_path.exists() else manual_path
    if not path.exists():
        raise FileNotFoundError(f"No concept CSV found for split={split}")

    df_raw = read_csv_flexible(path)
    id_col = infer_id_col(df_raw)
    img_map = build_image_map(split)

    rows, missing_path, no_cui = [], 0, 0
    for _, row in df_raw.iterrows():
        image_id_raw = str(row[id_col]).strip()
        key_candidates = [image_id_raw, Path(image_id_raw).stem, Path(image_id_raw).name]
        img_path = None
        for k in key_candidates:
            if k in img_map:
                img_path = img_map[k]
                break
        if img_path is None:
            missing_path += 1
            continue

        cuis = extract_cuis_from_row(row)
        if len(cuis) == 0:
            no_cui += 1
            continue

        rows.append({
            "split": split,
            "image_id": Path(img_path).stem,
            "path": img_path,
            "cuis": cuis,
            "n_cuis": len(cuis),
        })

    out = pd.DataFrame(rows)
    print(f"{split}: usable={len(out)} | missing_path={missing_path} | no_cui={no_cui} | csv={path.name}")
    return out

def sample_df(df, max_n, name):
    if max_n is None or len(df) <= max_n:
        return df.reset_index(drop=True)
    out = df.sample(n=max_n, random_state=CFG["seed"]).reset_index(drop=True)
    print(f"Sampled {name}: {len(df)} -> {len(out)}")
    return out

train_df = sample_df(load_split("train"), CFG["max_train_samples"], "train")
valid_df = sample_df(load_split("valid"), CFG["max_valid_samples"], "valid")
test_df  = sample_df(load_split("test"),  CFG["max_test_samples"],  "test")

all_df = pd.concat([train_df, valid_df, test_df], ignore_index=True)
summary = all_df.groupby("split").agg(images=("image_id", "count"), mean_cuis=("n_cuis", "mean"), median_cuis=("n_cuis", "median"))
display(summary)
summary.to_csv(os.path.join(OUT_DIR, "dataset_summary.csv"))

plt.figure(figsize=(5,4))
summary["images"].reindex(["train", "valid", "test"]).plot(kind="bar")
plt.title("Images per split")
plt.xlabel("Split")
plt.ylabel("Images")
savefig("01_images_per_split.png")


train_counter = Counter(c for cuis in train_df["cuis"] for c in cuis)
label_cuis = [c for c, n in train_counter.most_common() if n >= CFG["min_cui_freq"]]
label_cuis = label_cuis[:CFG["max_train_cuis"]]
label_to_idx = {c:i for i,c in enumerate(label_cuis)}
idx_to_label = {i:c for c,i in label_to_idx.items()}
n_labels = len(label_cuis)
print("Number of CUI labels used:", n_labels)

pd.DataFrame({"cui": label_cuis, "train_frequency": [train_counter[c] for c in label_cuis]}).to_csv(
    os.path.join(OUT_DIR, "cui_label_space.csv"), index=False
)

def add_label_indices(df):
    df = df.copy()
    inds = []
    kept_cuis = []
    for cuis in df["cuis"]:
        row_inds = sorted([label_to_idx[c] for c in cuis if c in label_to_idx])
        inds.append(row_inds)
        kept_cuis.append([idx_to_label[i] for i in row_inds])
    df["label_indices"] = inds
    df["kept_cuis"] = kept_cuis
    df = df[df["label_indices"].map(len) > 0].reset_index(drop=True)
    return df

train_df = add_label_indices(train_df)
valid_df = add_label_indices(valid_df)
test_df = add_label_indices(test_df)
print("After filtering to selected CUI label space:", len(train_df), len(valid_df), len(test_df))

freq_df = pd.DataFrame(train_counter.most_common(30), columns=["CUI", "frequency"])
plt.figure(figsize=(8,5))
plt.barh(freq_df["CUI"][::-1], freq_df["frequency"][::-1])
plt.title("Top 30 training CUIs")
plt.xlabel("Frequency")
plt.ylabel("CUI")
savefig("02_top_cui_frequencies.png")

plt.figure(figsize=(6,4))
plt.hist(all_df["n_cuis"], bins=30)
plt.title("CUI count per image")
plt.xlabel("Number of CUIs")
plt.ylabel("Images")
savefig("03_cui_count_histogram.png")


class RocoPathDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return row["path"], row["image_id"]

class FeatureDataset(Dataset):
    def __init__(self, features, label_indices, n_labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.label_indices = list(label_indices)
        self.n_labels = n_labels

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, idx):
        y = torch.zeros(self.n_labels, dtype=torch.float32)
        for j in self.label_indices[idx]:
            y[j] = 1.0
        return self.features[idx], y


def ensure_medimageinsight_package():
    try:
        from MedImageInsights.Models.medimageinsightmodel import MedImageInsight
        return MedImageInsight
    except Exception as e:
        print("MedImageInsights import failed before install:", repr(e))


    pip_install(["MedImageInsights==0.1.1", "--no-deps"])
    light_deps = [
        "huggingface_hub", "hf_xet", "einops", "fvcore", "yacs", "timm", "transformers",
        "safetensors", "tenacity", "mup"
    ]
    for dep in light_deps:
        try:
            pip_install([dep])
        except Exception as dep_e:
            print("Could not install", dep, "->", repr(dep_e))

    from MedImageInsights.Models.medimageinsightmodel import MedImageInsight
    return MedImageInsight

MedImageInsight = ensure_medimageinsight_package()

try:
    from huggingface_hub import snapshot_download
except Exception:
    pip_install(["huggingface_hub", "hf_xet"])
    from huggingface_hub import snapshot_download

MODEL_DIR_NAME_CANDIDATES = ["v20240927", "2024.09.27", "2024_09_27"]
VISION_WEIGHT_CANDIDATES = ["medimageinsight-v1.0.0.pt", "medimageinsigt-v1.0.0.pt"]
LANGUAGE_WEIGHT_CANDIDATES = ["language_model.pth"]

def find_candidate_model_dirs(base):
    base = Path(base)
    candidates = []
    if not base.exists():
        return candidates
    for p in [base] + list(base.rglob("config.yaml")):
        d = p.parent if p.name == "config.yaml" else p
        if (d / "config.yaml").exists():
            candidates.append(d)

    candidates = sorted(set(candidates), key=lambda x: (x.name not in MODEL_DIR_NAME_CANDIDATES, len(str(x))))
    return candidates

def locate_medimageinsight_model_dir():
    roots_to_check = []

    env_dir = os.environ.get("MEDIMAGEINSIGHT_MODEL_DIR")
    if env_dir:
        roots_to_check.append(env_dir)

    try:
        import MedImageInsights
        pkg_root = Path(MedImageInsights.__file__).resolve().parent
        roots_to_check += [pkg_root, pkg_root.parent]
    except Exception:
        pass

    roots_to_check += [MODEL_CACHE_DIR, os.environ.get("DATA_SEARCH_DIR", "../data"), os.environ.get("OUTPUT_ROOT", "../results")]

    for root in roots_to_check:
        for d in find_candidate_model_dirs(root):
            for weight_name in VISION_WEIGHT_CANDIDATES:
                if (d / "vision_model" / weight_name).exists():
                    return str(d), weight_name

    print("Downloading MedImageInsight model files from Hugging Face. This is large and requires Kaggle Internet ON.")
    local_dir = os.path.join(MODEL_CACHE_DIR, "lion-ai-MedImageInsights")
    snapshot_download(
        repo_id=CFG["hf_repo_id"],
        local_dir=local_dir,
        local_dir_use_symlinks=False,
        allow_patterns=["2024.09.27/**", "v20240927/**", "*.md"],
    )

    for d in find_candidate_model_dirs(local_dir):
        for weight_name in VISION_WEIGHT_CANDIDATES:
            if (d / "vision_model" / weight_name).exists():
                return str(d), weight_name


    dirs = find_candidate_model_dirs(local_dir)
    if dirs:
        return str(dirs[0]), VISION_WEIGHT_CANDIDATES[0]
    raise FileNotFoundError("Could not locate MedImageInsight config.yaml/model weights. Check Internet/model download.")

def _safe_link_or_copy(src, dst):
    """Create a symlink when possible; otherwise copy. Keeps large model files from being duplicated."""
    src = Path(src)
    dst = Path(dst)
    if dst.exists() or dst.is_symlink():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(str(src), str(dst), target_is_directory=src.is_dir())
    except Exception:
        if src.is_dir():
            shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
        else:
            shutil.copy2(str(src), str(dst))

def prepare_medimageinsight_compatible_dir(model_dir):
    """
    Fixes the common Kaggle/MedImageInsights loader error:
    HFValidationError: ... clip_tokenizer_v4162

    The Lion-AI Hugging Face repo stores the tokenizer folder as:
        language_model/clip_tokenizer_4.16.2

    Some pip-wrapper versions look for:
        language_model/clip_tokenizer_v4162

    This function creates a writable compatibility model folder with BOTH names,
    without copying the 2.4GB vision checkpoint when symlinks are allowed.
    """
    src_dir = Path(model_dir).resolve()
    official_tok = src_dir / "language_model" / "clip_tokenizer_4.16.2"
    wrapper_tok = src_dir / "language_model" / "clip_tokenizer_v4162"


    if wrapper_tok.exists():
        return str(src_dir)


    if official_tok.exists():
        try:
            _safe_link_or_copy(official_tok, wrapper_tok)
            if wrapper_tok.exists():
                print("Created tokenizer alias:", wrapper_tok)
                return str(src_dir)
        except Exception as e:
            print("Could not write alias inside model_dir; creating compatibility folder instead:", repr(e))


    if not official_tok.exists():
        print("Tokenizer folder missing. Downloading tokenizer files from Hugging Face...")
        dl_root = Path(MODEL_CACHE_DIR) / "lion-ai-MedImageInsights"
        snapshot_download(
            repo_id=CFG["hf_repo_id"],
            local_dir=str(dl_root),
            local_dir_use_symlinks=False,
            allow_patterns=[
                "2024.09.27/config.yaml",
                "2024.09.27/language_model/clip_tokenizer_4.16.2/**",
                "2024.09.27/vision_model/**",
            ],
        )
        downloaded_model_dir = dl_root / "2024.09.27"
        downloaded_tok = downloaded_model_dir / "language_model" / "clip_tokenizer_4.16.2"
        if downloaded_tok.exists():
            official_tok = downloaded_tok

            if (downloaded_model_dir / "vision_model" / vision_model_name).exists():
                src_dir = downloaded_model_dir
        else:
            raise FileNotFoundError(
                "Could not find MedImageInsight tokenizer folder. Expected "
                "'language_model/clip_tokenizer_4.16.2'. Turn Kaggle Internet ON and rerun."
            )


    compat_dir = Path(MODEL_CACHE_DIR) / "compatible_2024.09.27"
    compat_dir.mkdir(parents=True, exist_ok=True)


    if (src_dir / "config.yaml").exists():
        _safe_link_or_copy(src_dir / "config.yaml", compat_dir / "config.yaml")
    else:
        raise FileNotFoundError(f"Missing config.yaml in {src_dir}")

    if (src_dir / "vision_model").exists():
        _safe_link_or_copy(src_dir / "vision_model", compat_dir / "vision_model")
    else:
        raise FileNotFoundError(f"Missing vision_model folder in {src_dir}")


    (compat_dir / "language_model").mkdir(parents=True, exist_ok=True)
    _safe_link_or_copy(official_tok, compat_dir / "language_model" / "clip_tokenizer_4.16.2")
    _safe_link_or_copy(official_tok, compat_dir / "language_model" / "clip_tokenizer_v4162")

    print("Using MedImageInsight compatibility model_dir:", compat_dir)
    return str(compat_dir)

model_dir, vision_model_name = locate_medimageinsight_model_dir()
model_dir = prepare_medimageinsight_compatible_dir(model_dir)

print("MedImageInsight model_dir:", model_dir)
print("MedImageInsight vision_model_name:", vision_model_name)

classifier = MedImageInsight(model_dir=model_dir, vision_model_name=vision_model_name, language_model_name="language_model.pth")
classifier.load_model()
classifier.model.eval()
for p in classifier.model.parameters():
    p.requires_grad = False


def image_file_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def collate_paths(batch):
    paths, ids = zip(*batch)
    return list(paths), list(ids)

@torch.no_grad()
def extract_medimageinsight_features(df, split_name):
    feat_path = os.path.join(FEATURE_DIR, f"{split_name}_{CFG['model_tag']}_features.npy")
    id_path = os.path.join(FEATURE_DIR, f"{split_name}_{CFG['model_tag']}_ids.csv")

    if CFG["reuse_cached_features"] and os.path.exists(feat_path) and os.path.exists(id_path):
        feats = np.load(feat_path)
        ids = pd.read_csv(id_path)["image_id"].astype(str).tolist()
        if len(ids) == len(df):
            print(f"Loaded cached {split_name} MedImageInsight features:", feats.shape)
            return feats.astype("float32"), ids
        print(f"Cache length mismatch for {split_name}; recomputing.")

    loader = DataLoader(
        RocoPathDataset(df),
        batch_size=CFG["embed_batch_size"],
        shuffle=False,
        num_workers=CFG["num_workers"],
        pin_memory=False,
        collate_fn=collate_paths,
    )

    feats, ids = [], []
    classifier.model.eval()
    for step, (paths, image_ids) in enumerate(loader, 1):
        images_b64 = []
        good_ids = []
        for p, img_id in zip(paths, image_ids):
            try:
                images_b64.append(image_file_to_base64(p))
                good_ids.append(img_id)
            except Exception as e:
                print("Skipping unreadable image:", p, repr(e))

        if len(images_b64) == 0:
            continue


        out = classifier.encode(images=images_b64)
        emb = np.asarray(out["image_embeddings"], dtype="float32")
        if emb.ndim == 1:
            emb = emb[None, :]
        feats.append(emb)
        ids.extend(good_ids)

        if step % 25 == 0:
            print(f"{split_name}: encoded {len(ids)}/{len(df)}")

    feats = np.concatenate(feats, axis=0).astype("float32")
    np.save(feat_path, feats)
    pd.DataFrame({"image_id": ids}).to_csv(id_path, index=False)
    print(f"Saved {split_name} MedImageInsight features:", feats.shape)
    return feats, ids

train_feats, train_ids = extract_medimageinsight_features(train_df, "train")
valid_feats, valid_ids = extract_medimageinsight_features(valid_df, "valid")
num_features = int(train_feats.shape[1])
print("MedImageInsight feature dim:", num_features)


class CUIProjectionHead(nn.Module):
    def __init__(self, in_dim, proj_dim, n_labels, dropout=0.2):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.projector = nn.Sequential(
            nn.Linear(in_dim, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, proj_dim),
        )
        self.classifier = nn.Linear(proj_dim, n_labels)

    def forward(self, x, return_embedding=False):
        z = self.projector(self.norm(x))
        z = F.normalize(z, dim=1)
        if return_embedding:
            return z
        return self.classifier(z)

head = CUIProjectionHead(
    in_dim=num_features,
    proj_dim=CFG["projection_dim"],
    n_labels=n_labels,
    dropout=CFG["head_dropout"],
).to(device)

pos_counts = np.zeros(n_labels, dtype=np.float32)
for inds in train_df["label_indices"]:
    pos_counts[inds] += 1
pos_weight = (len(train_df) - pos_counts) / np.maximum(pos_counts, 1)
pos_weight = np.clip(pos_weight, 1.0, 20.0)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(head.parameters(), lr=CFG["head_lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CFG["epochs"]))
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
CKPT_PATH = os.path.join(OUT_DIR, f"best_{CFG['model_tag']}_cui_projection_head.pth")

train_loader = DataLoader(
    FeatureDataset(train_feats, train_df["label_indices"], n_labels),
    batch_size=CFG["batch_size"], shuffle=True,
    num_workers=0, pin_memory=True,
)
valid_loader = DataLoader(
    FeatureDataset(valid_feats, valid_df["label_indices"], n_labels),
    batch_size=CFG["batch_size"], shuffle=False,
    num_workers=0, pin_memory=True,
)

def run_one_epoch(loader, train=True):
    head.train(train)
    total, n = 0.0, 0
    for feats, y in loader:
        feats = feats.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = head(feats)
                loss = criterion(logits, y)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total += loss.item() * feats.size(0)
        n += feats.size(0)

    return total / max(n, 1)

history = []
best_val = float("inf")

if CFG["train"]:
    print(f"\nTraining CUI projection head for {CFG['model_tag']}...")
    for epoch in range(1, CFG["epochs"] + 1):
        t0 = time.time()
        tr_loss = run_one_epoch(train_loader, train=True)
        va_loss = run_one_epoch(valid_loader, train=False)
        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": tr_loss,
            "valid_loss": va_loss,
            "lr": optimizer.param_groups[0]["lr"],
            "seconds": time.time() - t0,
        }
        history.append(row)
        print(f"Epoch {epoch:02d}/{CFG['epochs']} | train={tr_loss:.5f} | valid={va_loss:.5f} | time={row['seconds']:.1f}s")

        if va_loss < best_val:
            best_val = va_loss
            torch.save({
                "head": head.state_dict(),
                "cfg": CFG,
                "label_to_idx": label_to_idx,
                "best_val": best_val,
                "num_features": num_features,
            }, CKPT_PATH)
            print("  saved best checkpoint")

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(os.path.join(OUT_DIR, "training_history.csv"), index=False)

    plt.figure(figsize=(6,4))
    plt.plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="train")
    plt.plot(hist_df["epoch"], hist_df["valid_loss"], marker="o", label="valid")
    plt.title("Training and validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("BCE loss")
    plt.legend()
    savefig("04_training_loss_curve.png")

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    head.load_state_dict(ckpt["head"])
    print("Loaded best checkpoint:", CKPT_PATH)


@torch.no_grad()
def project_features_to_embeddings(features, name):
    emb_path = os.path.join(OUT_DIR, f"{name}_embeddings.npy")
    if CFG["reuse_cached_features"] and os.path.exists(emb_path):
        embs = np.load(emb_path).astype("float32")
        if embs.shape[0] == features.shape[0]:
            print("Loaded cached projected embeddings:", name, embs.shape)
            return embs

    if not CFG["use_trained_projection_for_retrieval"]:
        embs = features / np.maximum(np.linalg.norm(features, axis=1, keepdims=True), 1e-8)
        np.save(emb_path, embs.astype("float32"))
        return embs.astype("float32")

    loader = DataLoader(torch.tensor(features, dtype=torch.float32), batch_size=CFG["batch_size"], shuffle=False)
    head.eval()
    embs = []
    for feats in loader:
        feats = feats.to(device, non_blocking=True)
        z = head(feats, return_embedding=True)
        embs.append(z.cpu().numpy().astype("float32"))
    embs = np.concatenate(embs, axis=0).astype("float32")
    np.save(emb_path, embs)
    print(name, embs.shape)
    return embs

split_to_df = {"train": train_df, "valid": valid_df, "test": test_df}
query_df = split_to_df[CFG["eval_split"]].reset_index(drop=True)
pool_df = split_to_df[CFG["retrieval_pool"]].reset_index(drop=True)

query_raw_feats, query_ids = extract_medimageinsight_features(query_df, CFG["eval_split"] + "_query")
if CFG["eval_split"] == CFG["retrieval_pool"] and len(query_df) == len(pool_df) and list(query_df["image_id"]) == list(pool_df["image_id"]):
    pool_raw_feats, pool_ids = query_raw_feats, query_ids
else:
    pool_raw_feats, pool_ids = extract_medimageinsight_features(pool_df, CFG["retrieval_pool"] + "_pool")

query_embs = project_features_to_embeddings(query_raw_feats, CFG["eval_split"] + "_query")
pool_embs = project_features_to_embeddings(pool_raw_feats, CFG["retrieval_pool"] + "_pool")

pd.DataFrame({"image_id": query_ids}).to_csv(os.path.join(OUT_DIR, f"{CFG['eval_split']}_query_ids.csv"), index=False)
pd.DataFrame({"image_id": pool_ids}).to_csv(os.path.join(OUT_DIR, f"{CFG['retrieval_pool']}_pool_ids.csv"), index=False)

same_pool = (
    CFG["eval_split"] == CFG["retrieval_pool"] and
    len(query_df) == len(pool_df) and
    list(query_df["image_id"]) == list(pool_df["image_id"])
)
print("same query/pool:", same_pool)


def compute_topk_by_embedding(query_embs, pool_embs, topk, chunk_size, same_pool):
    q = torch.tensor(query_embs, dtype=torch.float32, device=device)
    p = torch.tensor(pool_embs, dtype=torch.float32, device=device)
    all_idx, all_sim = [], []

    k_eff = min(topk, p.shape[0] - 1 if same_pool else p.shape[0])
    for s in range(0, q.shape[0], chunk_size):
        e = min(s + chunk_size, q.shape[0])
        sim = q[s:e] @ p.T

        if same_pool:
            diag = torch.arange(s, e, device=device)
            sim[torch.arange(e - s, device=device), diag] = -1e9

        vals, inds = torch.topk(sim, k=k_eff, dim=1)
        all_idx.append(inds.cpu().numpy())
        all_sim.append(vals.cpu().numpy())

    return np.vstack(all_idx), np.vstack(all_sim)

top_idx, top_sim = compute_topk_by_embedding(
    query_embs, pool_embs,
    topk=CFG["topk_max"],
    chunk_size=CFG["eval_chunk_size"],
    same_pool=same_pool,
)
np.save(os.path.join(OUT_DIR, "top_indices.npy"), top_idx)
np.save(os.path.join(OUT_DIR, "top_similarities.npy"), top_sim)
print("top_idx:", top_idx.shape, "top_sim:", top_sim.shape)

plt.figure(figsize=(6,4))
plt.hist(top_sim[:, 0], bins=50)
plt.title("Top-1 cosine similarity distribution")
plt.xlabel("Cosine similarity")
plt.ylabel("Queries")
savefig("05_top1_similarity_histogram.png")

plt.figure(figsize=(6,4))
plt.hist(top_sim.reshape(-1), bins=80)
plt.title("Top-K cosine similarity distribution")
plt.xlabel("Cosine similarity")
plt.ylabel("Retrieved pairs")
savefig("06_topk_similarity_histogram.png")


def label_sets_from_df(df):
    return [set(x) for x in df["label_indices"].tolist()]

query_sets = label_sets_from_df(query_df)
pool_sets = label_sets_from_df(pool_df)

def make_sparse_label_matrix(df, n_labels):
    rows, cols = [], []
    for i, inds in enumerate(df["label_indices"]):
        for j in inds:
            rows.append(i)
            cols.append(j)
    data = np.ones(len(rows), dtype=np.int8)
    return csr_matrix((data, (rows, cols)), shape=(len(df), n_labels), dtype=np.int8)

qY = make_sparse_label_matrix(query_df, n_labels)
pY = make_sparse_label_matrix(pool_df, n_labels)

def compute_total_relevant(qY, pY, same_pool, chunk_size=512):
    totals = []
    for s in range(0, qY.shape[0], chunk_size):
        e = min(s + chunk_size, qY.shape[0])
        rel = (qY[s:e] @ pY.T).astype(bool).astype(np.int8)
        if same_pool:
            for local_i, global_i in enumerate(range(s, e)):
                rel[local_i, global_i] = 0
        totals.extend(np.asarray(rel.sum(axis=1)).ravel().astype(int).tolist())
    return np.array(totals, dtype=int)

total_relevant = compute_total_relevant(qY, pY, same_pool=same_pool)
print("Mean relevant pool images per query:", float(total_relevant.mean()))


def jaccard(a, b):
    """Jaccard index between two label-index sets."""
    if len(a) == 0 and len(b) == 0:
        return 0.0
    return len(a & b) / max(len(a | b), 1)

def dcg_at_k(gains, k):
    """Standard DCG@K using gains[0..k-1] and log2 discount."""
    gains = np.asarray(gains[:k], dtype=np.float64)
    if len(gains) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(gains) + 2))
    return float(np.sum(gains / discounts))

def ndcg_at_k(predicted_gains, ideal_gains, k):
    """
    NDCG@K.
    predicted_gains : Jaccard scores of retrieved images in model-rank order.
    ideal_gains     : all pool Jaccard scores sorted descending (ideal ranking).
    """
    ideal_dcg = dcg_at_k(ideal_gains, k)
    if ideal_dcg <= 0.0:
        return np.nan
    return dcg_at_k(predicted_gains, k) / ideal_dcg

def average_precision_at_k(rel, total_rel, k):
    rel = np.array(rel[:k], dtype=np.float32)
    if total_rel <= 0:
        return np.nan
    denom = min(total_rel, k)
    if denom <= 0:
        return np.nan
    precisions, hits = [], 0
    for r, is_rel in enumerate(rel, start=1):
        if is_rel:
            hits += 1
            precisions.append(hits / r)
    return float(np.sum(precisions) / denom) if precisions else 0.0

def reciprocal_rank_at_k(rel, k):
    for r, is_rel in enumerate(rel[:k], start=1):
        if is_rel:
            return 1.0 / r
    return 0.0


per_query_rows = []
for qi in range(len(query_df)):
    qset = query_sets[qi]


    all_pool_jac = np.array([
        jaccard(qset, pool_sets[int(pi)]) if not (same_pool and pi == qi) else 0.0
        for pi in range(len(pool_df))
    ], dtype=np.float64)


    ideal_gains_sorted = np.sort(all_pool_jac)[::-1]


    retrieved = top_idx[qi]
    predicted_jac  = [all_pool_jac[int(pi)] for pi in retrieved]
    rel_full       = [all_pool_jac[int(pi)] > 0 for pi in retrieved]
    jac_full       = predicted_jac

    row = {
        "query_index": qi,
        "query_image_id": query_df.iloc[qi]["image_id"],
        "n_query_cuis": len(qset),
        "total_relevant_in_pool": int(total_relevant[qi]),
        "top1_similarity": float(top_sim[qi, 0]),
    }

    for k in CFG["ks_curve"]:
        kk = min(k, len(retrieved))
        rel = rel_full[:kk]

        retrieved_cuis = set()
        for pi in retrieved[:kk]:
            retrieved_cuis |= pool_sets[int(pi)]


        row[f"CUI@{k}"] = ndcg_at_k(predicted_jac, ideal_gains_sorted, kk)

        row[f"Precision@{k}"]   = float(np.mean(rel)) if kk > 0 else np.nan
        row[f"Recall@{k}"]      = len(qset & retrieved_cuis) / max(len(qset), 1)
        row[f"mAP@{k}"]         = average_precision_at_k(rel_full, int(total_relevant[qi]), kk)
        row[f"MRR@{k}"]         = reciprocal_rank_at_k(rel_full, kk)
        row[f"MeanJaccard@{k}"] = float(np.mean(jac_full[:kk])) if kk > 0 else np.nan

    per_query_rows.append(row)

per_query_df = pd.DataFrame(per_query_rows)
per_query_df.to_csv(os.path.join(OUT_DIR, "per_query_metrics.csv"), index=False)

metric_rows = []
metrics = ["CUI", "Precision", "Recall", "mAP", "MRR", "MeanJaccard"]
for metric in metrics:
    for k in CFG["ks_curve"]:
        col = f"{metric}@{k}"
        vals = per_query_df[col].dropna().values.astype(float)
        if len(vals) == 0:
            continue
        mean  = float(np.mean(vals))
        std   = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        ci95  = float(1.96 * std / math.sqrt(len(vals))) if len(vals) > 1 else 0.0
        metric_rows.append({
            "metric": metric,
            "K": k,
            "mean": mean,
            "std": std,
            "ci95": ci95,
            "mean±std": f"{mean:.4f} ± {std:.4f}",
            "mean±95CI": f"{mean:.4f} ± {ci95:.4f}",
            "n_queries": len(vals),
        })

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(os.path.join(OUT_DIR, "retrieval_metrics_by_k.csv"), index=False)
main_metrics = metrics_df[metrics_df["K"].isin(CFG["ks_main"])].copy()
main_metrics.to_csv(os.path.join(OUT_DIR, "main_retrieval_metrics.csv"), index=False)
display(main_metrics.pivot(index="metric", columns="K", values="mean±std"))


for metric in ["CUI", "Precision", "Recall", "mAP", "MRR", "MeanJaccard"]:
    temp = metrics_df[metrics_df["metric"] == metric]
    plt.figure(figsize=(6,4))
    plt.plot(temp["K"], temp["mean"], marker="o")
    plt.fill_between(temp["K"], temp["mean"] - temp["ci95"], temp["mean"] + temp["ci95"], alpha=0.2)
    plt.title(f"{metric}@K")
    plt.xlabel("K")
    plt.ylabel(metric)
    plt.grid(True, alpha=0.3)
    savefig(f"07_{metric.lower()}_at_k_curve.png")

plt.figure(figsize=(6,4))
for metric in ["CUI", "Precision", "Recall", "mAP", "MRR", "MeanJaccard"]:
    temp = metrics_df[metrics_df["metric"] == metric]
    plt.plot(temp["K"], temp["mean"], marker="o", label=metric)
plt.title("Retrieval metrics across K")
plt.xlabel("K")
plt.ylabel("Mean score")
plt.legend()
plt.grid(True, alpha=0.3)
savefig("08_all_metrics_at_k.png")

plt.figure(figsize=(6,4))
prec = metrics_df[metrics_df["metric"] == "Precision"].sort_values("K")
rec = metrics_df[metrics_df["metric"] == "Recall"].sort_values("K")
plt.plot(rec["mean"].values, prec["mean"].values, marker="o")
for k, x, y in zip(rec["K"].values, rec["mean"].values, prec["mean"].values):
    if k in CFG["ks_main"]:
        plt.text(x, y, f"K={k}")
plt.title("Precision-Recall curve over K")
plt.xlabel("Recall@K")
plt.ylabel("Precision@K")
plt.grid(True, alpha=0.3)
savefig("09_precision_recall_curve.png")

plt.figure(figsize=(7,5))
metric_main = main_metrics.pivot(index="metric", columns="K", values="mean").reindex(["CUI", "Precision", "Recall", "mAP", "MRR", "MeanJaccard"])
metric_main.plot(kind="bar", figsize=(9,5))
plt.title("Main retrieval metrics")
plt.ylabel("Mean score")
plt.xticks(rotation=30, ha="right")
savefig("10_main_metrics_barplot.png")


pca_n = min(3000, len(query_embs))
if pca_n >= 10:
    idx = np.random.RandomState(CFG["seed"]).choice(len(query_embs), size=pca_n, replace=False)
    X = query_embs[idx]
    X2 = PCA(n_components=2, random_state=CFG["seed"]).fit_transform(X)
    n_clusters = min(8, max(2, pca_n // 200))
    clusters = KMeans(n_clusters=n_clusters, random_state=CFG["seed"], n_init=10).fit_predict(X)
    plt.figure(figsize=(7,6))
    plt.scatter(X2[:,0], X2[:,1], c=clusters, s=8, alpha=0.75)
    plt.title("PCA of MedImageInsight retrieval embeddings")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    savefig("11_pca_embeddings.png")


def open_img(path):
    return Image.open(path).convert("RGB")

def make_retrieval_grid(query_indices, name, top_n=5):
    query_indices = list(query_indices)
    if len(query_indices) == 0:
        return
    rows = len(query_indices)
    cols = top_n + 1
    plt.figure(figsize=(cols * 2.2, rows * 2.2))
    for r, qi in enumerate(query_indices):
        qrow = query_df.iloc[qi]
        qset = query_sets[qi]
        ax = plt.subplot(rows, cols, r * cols + 1)
        ax.imshow(open_img(qrow["path"]))
        ax.set_title("Query")
        ax.axis("off")
        for c in range(top_n):
            pi = int(top_idx[qi, c])
            prow = pool_df.iloc[pi]
            pset = pool_sets[pi]
            hit = len(qset & pset) > 0
            jac = jaccard(qset, pset)
            ax = plt.subplot(rows, cols, r * cols + c + 2)
            ax.imshow(open_img(prow["path"]))
            ax.set_title(f"Top {c+1} {'✓' if hit else '×'}\nJ={jac:.2f}")
            ax.axis("off")
    savefig(name)

rng = np.random.RandomState(CFG["seed"])
random_q = rng.choice(len(query_df), size=min(5, len(query_df)), replace=False)
make_retrieval_grid(random_q, "12_random_retrieval_grid.png", top_n=5)

if "Precision@10" in per_query_df.columns:
    best_q = per_query_df.sort_values(["Precision@10", "MRR@10", "top1_similarity"], ascending=False).head(5)["query_index"].values
    worst_q = per_query_df.sort_values(["Precision@10", "MRR@10", "top1_similarity"], ascending=True).head(5)["query_index"].values
    make_retrieval_grid(best_q, "13_best_retrieval_grid.png", top_n=5)
    make_retrieval_grid(worst_q, "14_worst_retrieval_grid.png", top_n=5)


rows = []
for qi in range(len(query_df)):
    qset = query_sets[qi]
    for rank in range(min(CFG["topk_max"], top_idx.shape[1])):
        pi = int(top_idx[qi, rank])
        pset = pool_sets[pi]
        rows.append({
            "query_index": qi,
            "query_image_id": query_df.iloc[qi]["image_id"],
            "rank": rank + 1,
            "retrieved_index": pi,
            "retrieved_image_id": pool_df.iloc[pi]["image_id"],
            "similarity": float(top_sim[qi, rank]),
            "is_cui_hit": int(len(qset & pset) > 0),
            "jaccard": jaccard(qset, pset),
            "shared_cuis": "|".join([idx_to_label[x] for x in sorted(qset & pset)]),
        })

topk_df = pd.DataFrame(rows)
topk_df.to_csv(os.path.join(OUT_DIR, "topk_retrieval_details.csv"), index=False)

with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump(CFG, f, indent=2)


zip_path = os.path.join(os.environ.get("OUTPUT_ROOT", "../results"), "medimageinsight_roco_retrieval_figures.zip")
if os.path.exists(zip_base + ".zip"):
    os.remove(zip_base + ".zip")
shutil.make_archive(zip_base, "zip", FIG_DIR)
print("\nDone.")
print("Output directory:", OUT_DIR)
print("Figures directory:", FIG_DIR)
print("Figures zip:", zip_base + ".zip")
print("Main metrics CSV:", os.path.join(OUT_DIR, "main_retrieval_metrics.csv"))
print("Top-k details CSV:", os.path.join(OUT_DIR, "topk_retrieval_details.csv"))
